In [0]:
import io
import unicodedata
import re
import logging
import pandas as pd

from pyspark.sql.functions import current_timestamp, lit

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# ── Pipeline configuration ────────────────────────────────────────────────────
WRITE_MODE    = "overwrite"          # "overwrite" or "append"
TARGET_SCHEMA = "sandbox_bronze_ss2" # must match the schema created in schema_creation.py
SOURCE_SYSTEM = "Local File (Databricks Volume)"  # lineage tag appended to every ingested row

# WHO export format: number of metadata lines before the real header row
# (Region Code, Region Name, Country Code, Country Name, Export date,
# Source location, "Metadata information:", blank line -> 8 lines)
METADATA_LINES = 8

# ── Files to ingest ───────────────────────────────────────────────────────────
# How to upload a file to Databricks before running this script:
#   1. Go to Catalog in the left menu.
#   2. Navigate to your catalog → schema → select or create a Volume.
#   3. Click "Upload to this Volume" and upload your CSV file.
#   4. Copy the full path shown (e.g. /Volumes/my_catalog/my_schema/landing/file.csv)
#      and paste it in `volume_path` below.
#
# Each entry is an INDEPENDENT source -> 1 bronze table each, no relationships.
LOCAL_FILES = [
    {
        "table_suffix": "guatemala",   # appended to table name: who_<table_suffix>
        "volume_path": "/Volumes/ss2_dw_workspace/sandbox_bronze_ss2/oms_local/who_mortality_db_Guatemala.csv",
        "file_name": "who_mortality_db_Guatemala.csv",  # used only for lineage metadata column
    },
    # Add more files here following the same pattern:
    # {
    #     "table_suffix": "costa_rica",
    #     "volume_path": "/Volumes/ss2_dw_workspace/sandbox_bronze_ss2/oms_local/who_mortality_db_CostaRica.csv",
    #     "file_name": "who_mortality_db_CostaRica.csv",
    # },
]


# ── Helpers (identical to ingesta_oms_databricks.py for consistency) ──────────

def clean_name(name):
    """Normalize table/column names to snake_case; strips .csv extension if present."""
    name = name.lower().replace(".csv", "")
    # ñ must be transliterated before NFKD decomposition
    name = name.replace("ñ", "ni")
    # Strip diacritical marks from accented vowels (á→a, é→e, ó→o, ú→u, etc.)
    name = unicodedata.normalize("NFKD", name)
    name = "".join(c for c in name if not unicodedata.combining(c))
    name = re.sub(r'[^a-z0-9_]', '_', name)
    name = re.sub(r'_+', '_', name)
    return name.strip('_')


def dedupe_columns(cols):
    """After normalization, distinct source columns can collide; append a counter to duplicates."""
    seen = {}
    result = []
    for col in cols:
        count = seen.get(col, 0)
        seen[col] = count + 1
        result.append(col if count == 0 else f"{col}_{count + 1}")
    return result


def read_who_csv_from_volume(volume_path, metadata_lines):
    """
    Reads a WHO-format CSV directly from a Databricks Volume path.
    Skips the metadata header lines and normalizes column names,
    identical to the Google Drive version but reading from the local
    Volume filesystem instead of downloading from the network.
    """
    # Unity Catalog Volume paths are POSIX paths, readable with plain Python
    # I/O. We read into an in-memory buffer so pandas handles encoding and
    # delimiter detection the same way it does with the BytesIO buffer from Drive.
    with open(volume_path, "rb") as f:
        buffer = io.BytesIO(f.read())
        # index_col=False prevents pandas from treating the first column as an
        # index when data rows have one more field than the header (every WHO
        # export row ends with a trailing comma), which would otherwise shift
        # every column one position to the right and silently drop the index.
        try:
            df = pd.read_csv(buffer, skiprows=metadata_lines, sep=None, engine="python", index_col=False)
        except Exception:
            # Fallback for files with malformed quoting
            buffer.seek(0)
            df = pd.read_csv(
                buffer,
                skiprows=metadata_lines,
                sep=None,
                engine="python",
                index_col=False,
                on_bad_lines="skip",
            )

    # Log raw column names so schema drift is immediately visible in the job log
    logger.info(f"    Raw schema ({len(df.columns)} cols): {list(df.columns)}")
    df.columns = dedupe_columns([clean_name(c) for c in df.columns])
    # Drop columns that pandas auto-generates from trailing delimiters
    df = df.loc[:, ~df.columns.str.match(r'^unnamed_\d+$')]
    logger.info(f"    Clean schema ({len(df.columns)} cols): {list(df.columns)}")
    return df


def ingest_local_file(table_suffix, volume_path, file_name, schema, write_mode, source_system, metadata_lines):
    """
    Full bronze ingestion for a single local file:
      Volume path → pandas DataFrame → Spark DataFrame → Delta table.
    Mirrors ingest_gdrive_file() from ingesta_oms_databricks.py exactly.
    """
    table_name = f"{schema}.who_{clean_name(table_suffix)}"
    logger.info("-" * 60)
    logger.info(f"Source file  : {file_name}")
    logger.info(f"Volume path  : {volume_path}")
    logger.info(f"Target table : {table_name}  [mode={write_mode}]")

    try:
        logger.info(f"  Reading: {file_name}")
        df = read_who_csv_from_volume(volume_path, metadata_lines)
    except Exception as exc:
        # Log and skip so one bad file doesn't abort the entire pipeline
        logger.warning(f"  Skipped {file_name}: {exc}")
        return

    if df.empty:
        logger.warning(f"  No valid data in '{file_name}', skipping write.")
        return

    # Resolve object columns with mixed types (int rows + str rows) that Arrow cannot infer
    for col in df.select_dtypes(include="object").columns:
        coerced = pd.to_numeric(df[col], errors="coerce")
        non_null = df[col].notna().sum()
        if non_null > 0 and coerced.notna().sum() / non_null >= 0.9:
            # Column is predominantly numeric → keep as float (NaN-compatible)
            df[col] = coerced
        else:
            # Text column → cast to str so Arrow maps to StringType uniformly
            df[col] = df[col].where(df[col].isna(), df[col].astype(str))

    sdf = spark.createDataFrame(df)
    sdf = (
        sdf
        .withColumn("ingestion_timestamp", current_timestamp())
        .withColumn("source_system",       lit(source_system))
        .withColumn("source_file",         lit(file_name))
        .withColumn("volume_path",         lit(volume_path))   # replaces gdrive_file_id
        .withColumn("country",             lit(table_suffix))
    )

    (
        sdf.write
        .format("delta")
        .mode(write_mode)
        .option("mergeSchema", "true")  # allows new columns across runs without schema errors
        .saveAsTable(table_name)
    )

    logger.info(f"  Wrote {len(df):,} rows → {table_name}")


# ── Entry point ───────────────────────────────────────────────────────────────
try:
    for config in LOCAL_FILES:
        ingest_local_file(
            table_suffix   = config["table_suffix"],
            volume_path    = config["volume_path"],
            file_name      = config["file_name"],
            schema         = TARGET_SCHEMA,
            write_mode     = WRITE_MODE,
            source_system  = SOURCE_SYSTEM,
            metadata_lines = METADATA_LINES,
        )

    logger.info("Pipeline completed successfully.")
except Exception as e:
    logger.error(f"Pipeline failed: {e}")
    raise  # re-raise so the Databricks job status reflects the failure